In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold

# 1. Load Dataset

In [2]:
# Load train and test data 
BASE_DIR = Path("../data")
train = pd.read_csv(BASE_DIR / "train.csv")
test  = pd.read_csv(BASE_DIR / "test.csv")

# Re-parse kolom "Date"
train["Date"] = pd.to_datetime(train["Date"])
test["Date"]  = pd.to_datetime(test["Date"])

print(f"\ntrain: {train.shape}")
print(f"test : {test.shape}")
train.head()


train: (1866532, 10)
test : (465999, 10)


,Unnamed: 0,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,Target
0,0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,0.000730
1,1,20170104_7412,2017-01-04,7412,719.0,725.0,719.0,721.0,201400,0.000000
2,2,20170104_7408,2017-01-04,7408,2459.0,2518.0,2447.0,2500.0,110900,0.004421
3,3,20170104_7315,2017-01-04,7315,465.0,494.0,465.0,493.0,41100,-0.004032
4,4,20170104_7313,2017-01-04,7313,3055.0,3150.0,3045.0,3135.0,248600,-0.009693


In [3]:
#split data into train and validation sets
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
    print(f"Fold {fold}: train={len(train_idx)} rows, val={len(val_idx)} rows")

# ambil fold terakhir sebagai train/validation final
train_idx, val_idx = list(tscv.split(train))[-1]

train_sub = train.iloc[train_idx].reset_index(drop=True)
val_sub   = train.iloc[val_idx].reset_index(drop=True)

print(f"\nTrain final : {train_sub.shape}")
print(f"Validation  : {val_sub.shape}")

Fold 0: train=311092 rows, val=311088 rows
Fold 1: train=622180 rows, val=311088 rows
Fold 2: train=933268 rows, val=311088 rows
Fold 3: train=1244356 rows, val=311088 rows
Fold 4: train=1555444 rows, val=311088 rows

Train final : (1555444, 10)
Validation  : (311088, 10)


# Preprocessing

In [4]:
# drop missing values
train_sub = train_sub.dropna().reset_index(drop=True)
val_sub   = val_sub.dropna().reset_index(drop=True)
test      = test.dropna().reset_index(drop=True)

In [5]:
# Outlier Handling (Data Mentah) — Pembersihan Nilai Ekstrem & Error Input
def clean_raw_outliers(df, name="data"):
    """
    Membersihkan data mentah dari nilai ekstrem dan error input:
    1. Harga non-positif (harga <= 0 atau bernilai negatif) pada Open, High, Low, Close.
    2. Volume negatif (Volume < 0).
    3. Inkonsistensi struktural OHLC akibat typo/kesalahan input (High < Low, High < Open/Close, Low > Open/Close).
    4. Anomali lonjakan harga intraday ekstrem akibat kesalahan ketik desimal (High / Low > 2.5).
    """
    before_len = len(df)
    
    # 1. Validasi nilai positif untuk harga dan non-negatif untuk volume
    price_cols = ["Open", "High", "Low", "Close"]
    valid_price = (df[price_cols] > 0).all(axis=1)
    valid_volume = df["Volume"] >= 0
    
    # 2. Validasi konsistensi relasi OHLC (mencegah typo/error pencatatan)
    valid_ohlc = (
        (df["High"] >= df["Low"]) &
        (df["High"] >= df["Open"]) &
        (df["High"] >= df["Close"]) &
        (df["Low"] <= df["Open"]) &
        (df["Low"] <= df["Close"])
    )
    
    # 3. Validasi batas rasio harian ekstrem (High / Low) untuk mendeteksi typo nilai
    # Pada bursa saham (JPX), pergerakan harian dibatasi batas auto-rejection (~15-30%)
    price_ratio = df["High"] / df["Low"]
    valid_ratio = price_ratio <= 2.5
    
    # Gabungkan seluruh filter validitas
    valid_mask = valid_price & valid_volume & valid_ohlc & valid_ratio
    anomalies_count = int((~valid_mask).sum())
    
    cleaned_df = df.loc[valid_mask].reset_index(drop=True)
    
    print(f"[{name}] Pembersihan Outlier Mentah & Input Error:")
    print(f"  - Total baris sebelum   : {before_len:,}")
    print(f"  - Anomali/error dibuang : {anomalies_count:,} baris")
    print(f"  - Baris valid tersisa   : {len(cleaned_df):,}")
    
    return cleaned_df

# Terapkan pembersihan pada data mentah sebelum feature engineering
train_sub = clean_raw_outliers(train_sub, "train_sub")
val_sub   = clean_raw_outliers(val_sub, "val_sub")
test      = clean_raw_outliers(test, "test")


[train_sub] Pembersihan Outlier Mentah & Input Error:
  - Total baris sebelum   : 1,551,801
  - Anomali/error dibuang : 0 baris
  - Baris valid tersisa   : 1,551,801
[val_sub] Pembersihan Outlier Mentah & Input Error:
  - Total baris sebelum   : 308,228
  - Anomali/error dibuang : 0 baris
  - Baris valid tersisa   : 308,228
[test] Pembersihan Outlier Mentah & Input Error:
  - Total baris sebelum   : 464,894
  - Anomali/error dibuang : 0 baris
  - Baris valid tersisa   : 464,894


# Feature Engineering

In [6]:
train_sub = train_sub.copy()
val_sub   = val_sub.copy()
test      = test.copy()

train_sub["_split"] = "train_sub"
val_sub["_split"]   = "val_sub"
test["_split"]      = "test"
full = pd.concat([train_sub, val_sub, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)

g = full.groupby("SecuritiesCode", sort=False)

# 1. Fitur Temporal
full["year"]       = full["Date"].dt.year
full["month"]      = full["Date"].dt.month
full["day"]        = full["Date"].dt.day
full["dayofweek"]  = full["Date"].dt.dayofweek
full["quarter"]    = full["Date"].dt.quarter
full["dayofyear"]  = full["Date"].dt.dayofyear
full["weekofyear"] = full["Date"].dt.isocalendar().week.astype(int)

# 2. Fitur Return (N-day)
full["ret_1d"]  = g["Close"].pct_change()
full["ret_5d"]  = g["Close"].pct_change(5)
full["ret_20d"] = g["Close"].pct_change(20)

# 3. Fitur Lag (explicit shift)
full["ret_1d_lag1"] = g["ret_1d"].shift(1)
full["ret_1d_lag2"] = g["ret_1d"].shift(2)
full["ret_1d_lag3"] = g["ret_1d"].shift(3)

# 4. Fitur Rolling Window (agregasi statistik)
full["vol_5d"]  = g["ret_1d"].transform(lambda s: s.rolling(5,  min_periods=2).std())
full["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())

vol_ma20             = g["Volume"].transform(lambda s: s.rolling(20, min_periods=1).mean())
full["volume_ratio"] = full["Volume"] / vol_ma20
del vol_ma20

# 5. Fitur Bentuk Intraday
full["intraday_range"] = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]     = (full["Close"] - full["Open"]) / full["Open"]
full["high_close"]     = (full["High"] - full["Close"]) / full["Close"]
full["low_close"]      = (full["Close"] - full["Low"]) / full["Close"]

# 6. Fitur Volume
full["log_volume"] = np.log1p(full["Volume"])

# 7. Fitur Cross-sectional Rank
full["ret_rank"]    = full.groupby("Date")["ret_1d"].rank(pct=True)
full["volume_rank"] = full.groupby("Date")["Volume"].rank(pct=True)

# pisahkan kembali
train_sub = full[full["_split"] == "train_sub"].drop(columns="_split").reset_index(drop=True)
val_sub   = full[full["_split"] == "val_sub"].drop(columns="_split").reset_index(drop=True)
test      = full[full["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

print("Fitur ditambahkan: Temporal, Return, Lag, Rolling Window, Intraday, Volume, Rank.")
print(f"train_sub: {train_sub.shape}, val_sub: {val_sub.shape}, test: {test.shape}")

Fitur ditambahkan: Temporal, Return, Lag, Rolling Window, Intraday, Volume, Rank.
train_sub: (1551801, 33), val_sub: (308228, 33), test: (464894, 33)


In [7]:
#outlier handling :capping
from math import isnan

def handle_outliers(df, column, method="iqr", action="cap", bounds=None):
    if column not in df.columns:
        raise KeyError(f"Column tidak ditemukan: {column}")
    if not pd.api.types.is_numeric_dtype(df[column]):
        raise TypeError(f"Column harus numerik: {column}")
    if method not in {"iqr", "zscore"}:
        raise ValueError("method harus 'iqr' atau 'zscore'")
    if action not in {"remove", "cap"}:
        raise ValueError("action harus 'remove' atau 'cap'")

    result = df.copy()
    values = result[column]

    if bounds is None:
        if method == "iqr":
            q1, q3 = values.quantile([0.25, 0.75])
            spread = q3 - q1
            lower = q1 - 1.5 * spread
            upper = q3 + 1.5 * spread
        else:
            mean = values.mean()
            std = values.std()
            lower = mean - 3 * std
            upper = mean + 3 * std
    else:
        lower, upper = bounds

    outlier_mask = values.notna() & ((values < lower) | (values > upper))
    before_rows = len(result)
    outlier_count = int(outlier_mask.sum())

    if action == "remove":
        result = result.loc[~outlier_mask].reset_index(drop=True)
        changed = outlier_count
        print(f"[{column}] {method} removal: {outlier_count} baris dihapus")
    else:
        result[column] = values.clip(lower=lower, upper=upper)
        changed = int((result[column] != values).fillna(False).sum())
        print(f"[{column}] {method} capping: {changed} nilai dibatasi")

    print(f"  Sebelum: {before_rows:,} baris | Sesudah: {len(result):,} baris")
    print(f"  Batas: [{lower:.6g}, {upper:.6g}] | Outlier terdeteksi: {outlier_count:,}")
    return result, (lower, upper)


# --- Terapkan ke SEMUA kolom skewed: train_sub -> hitung bounds, val_sub & test -> pakai bounds train_sub ---
CAP_COLS = [
    "ret_1d", "ret_5d", "ret_20d",
    "ret_1d_lag1", "ret_1d_lag2", "ret_1d_lag3",
    "vol_5d", "vol_20d",
    "volume_ratio",
]

cap_bounds = {}
for col in CAP_COLS:
    train_sub, bounds = handle_outliers(train_sub, column=col, method="iqr", action="cap")
    cap_bounds[col] = bounds

for col in CAP_COLS:
    val_sub, _ = handle_outliers(val_sub, column=col, method="iqr", action="cap", bounds=cap_bounds[col])
    test, _    = handle_outliers(test, column=col, method="iqr", action="cap", bounds=cap_bounds[col])

print("\nSemua kolom skewed sudah di-cap (bounds dari train_sub):")
for col, (lo, hi) in cap_bounds.items():
    print(f"  {col:15s}: [{lo:.6g}, {hi:.6g}]")

[ret_1d] iqr capping: 109350 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403926, 0.0402078] | Outlier terdeteksi: 107,376
[ret_5d] iqr capping: 117121 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0963203, 0.0970418] | Outlier terdeteksi: 107,251
[ret_20d] iqr capping: 130836 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.208932, 0.211786] | Outlier terdeteksi: 91,356
[ret_1d_lag1] iqr capping: 111129 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403739, 0.0401712] | Outlier terdeteksi: 107,181
[ret_1d_lag2] iqr capping: 112987 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403688, 0.0401496] | Outlier terdeteksi: 107,065
[ret_1d_lag3] iqr capping: 114797 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403737, 0.0401183] | Outlier terdeteksi: 106,901
[vol_5d] iqr capping:

In [ ]:
#security code target encoding
mean_target = train_sub.groupby("SecuritiesCode")["Target"].mean()
global_mean = train_sub["Target"].mean()

train_sub["SecuritiesCode_te"] = train_sub["SecuritiesCode"].map(mean_target)
val_sub["SecuritiesCode_te"]   = val_sub["SecuritiesCode"].map(mean_target).fillna(global_mean)
test["SecuritiesCode_te"]      = test["SecuritiesCode"].map(mean_target).fillna(global_mean)

print(f"SecuritiesCode_te dibuat. mean (train_sub)={train_sub['SecuritiesCode_te'].mean():.6f}")
print(f"SecuritiesCode_te dibuat. mean (val_sub)={val_sub['SecuritiesCode_te'].mean():.6f}")
print(f"SecuritiesCode_te dibuat. mean (test)={test['SecuritiesCode_te'].mean():.6f}")

In [8]:
# Finalisasi dataset untuk train_sub, val_sub, dan test
# Kolom OHLCV tidak digunakan sebagai fitur model setelah feature engineering.
feature_drop = ["Open", "High", "Low", "Close", "Volume"]
identifier_drop = ["Unnamed: 0", "RowId", "SecuritiesCode", "Date"]

train_final = train_sub.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)
val_final   = val_sub.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)
test_final  = test.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)

train_X = train_final.drop(columns="Target", errors="ignore")
train_y = train_final["Target"]
val_X   = val_final.drop(columns="Target", errors="ignore")
val_y   = val_final["Target"]
test_X  = test_final.drop(columns="Target", errors="ignore")
test_y  = test_final["Target"]                      

train_X = train_X.drop(columns=identifier_drop, errors="ignore")
val_X   = val_X.drop(columns=identifier_drop, errors="ignore")
test_X  = test_X.drop(columns=identifier_drop, errors="ignore")

print(f"Fitur final ({len(train_X.columns)} kolom): {list(train_X.columns)}")
print(f"train_sub: {train_X.shape}, target: {train_y.shape}")
print(f"val_sub  : {val_X.shape}, target: {val_y.shape}")
print(f"test     : {test_X.shape}, target: {test_y.shape}")   

Fitur final (23 kolom): ['year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear', 'ret_1d', 'ret_5d', 'ret_20d', 'ret_1d_lag1', 'ret_1d_lag2', 'ret_1d_lag3', 'vol_5d', 'vol_20d', 'volume_ratio', 'intraday_range', 'close_open', 'high_close', 'low_close', 'log_volume', 'ret_rank', 'volume_rank']
train_sub: (1512321, 23), target: (1512321,)
val_sub  : (307872, 23), target: (307872,)
test     : (464730, 23), target: (464730,)


In [9]:
# Feature scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

FEATURE_COLS = train_X.columns.tolist()
TARGET_COL = "Target"

X_train = train_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_train = train_y.to_numpy(dtype=np.float64).reshape(-1, 1)
X_val   = val_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_val   = val_y.to_numpy(dtype=np.float64).reshape(-1, 1)
X_test  = test_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_test  = test_y.to_numpy(dtype=np.float64).reshape(-1, 1)      

# Pipeline Machine Learning
scaler_X_svr = StandardScaler()
scaler_y_svr = StandardScaler()
X_train_svr = scaler_X_svr.fit_transform(X_train)
X_val_svr   = scaler_X_svr.transform(X_val)
X_test_svr  = scaler_X_svr.transform(X_test)
y_train_svr = scaler_y_svr.fit_transform(y_train).ravel()
y_val_svr   = scaler_y_svr.transform(y_val).ravel()
y_test_svr  = scaler_y_svr.transform(y_test).ravel()             

# Pipeline Deep Learning
scaler_X_lstm = MinMaxScaler()
scaler_y_lstm = MinMaxScaler()
X_train_lstm = scaler_X_lstm.fit_transform(X_train)
X_val_lstm   = scaler_X_lstm.transform(X_val)
X_test_lstm  = scaler_X_lstm.transform(X_test)
y_train_lstm = scaler_y_lstm.fit_transform(y_train).ravel()
y_val_lstm   = scaler_y_lstm.transform(y_val).ravel()
y_test_lstm  = scaler_y_lstm.transform(y_test).ravel()           

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

Jumlah fitur: 23
X_train: (1512321, 23) | X_val: (307872, 23) | X_test: (464730, 23)


# Modelling

In [ ]:
%pip install tensorflow

In [ ]:
# Build LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

X_train_lstm_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_lstm_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_lstm_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

lstm_model = Sequential([
    Input(shape=(1, n_features)),
    LSTM(50, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mean_squared_error")
lstm_model.summary()

print(f"X_train_lstm: {X_train_lstm_3d.shape}")
print(f"X_val_lstm  : {X_val_lstm_3d.shape}")
print(f"X_test_lstm : {X_test_lstm_3d.shape}")

In [ ]:
# training model
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training LSTM pada {len(X_train_lstm_3d):,} sample train "
      f"dan {len(X_val_lstm_3d):,} sample validation (maks 100 epoch) ...")

history = lstm_model.fit(
    X_train_lstm_3d, y_train_lstm,              
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=16,
    epochs=100,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False,
)

In [ ]:
# evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Generate predictions on the test set in the scaled target space.
y_pred_lstm_scaled = lstm_model.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_lstm_scaled = y_test_lstm.ravel()

# Inverse transform back to the original target scale for meaningful metrics.
y_pred_lstm = scaler_y_lstm.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).ravel()
y_true_lstm = scaler_y_lstm.inverse_transform(y_true_lstm_scaled.reshape(-1, 1)).ravel()

mse_lstm = mean_squared_error(y_true_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = mean_absolute_error(y_true_lstm, y_pred_lstm)

print(f"LSTM -> MSE: {mse_lstm:.8f} | RMSE: {rmse_lstm:.8f} | MAE: {mae_lstm:.8f}")

# feature selection

In [ ]:
# Feature Selection - SelectKBest (f_regression), efisien untuk dataset besar
from sklearn.feature_selection import SelectKBest, f_regression

k = 15  # jumlah fitur yang ingin dipertahankan, sesuaikan kebutuhan

selector = SelectKBest(score_func=f_regression, k=k)
selector.fit(train_X, train_y)

feature_scores = pd.Series(selector.scores_, index=train_X.columns).sort_values(ascending=False)
SELECTED_FEATURES = train_X.columns[selector.get_support()].tolist()

print(f"Total fitur awal    : {train_X.shape[1]}")
print(f"Fitur terpilih (k={k}): {len(SELECTED_FEATURES)}")
print(f"\nTop {k} fitur (F-score tertinggi):")
print(feature_scores.head(k))

# Terapkan pilihan fitur yang sama ke train/val/test
train_X_sel = train_X[SELECTED_FEATURES]
val_X_sel   = val_X[SELECTED_FEATURES]
test_X_sel  = test_X[SELECTED_FEATURES]

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_scores.head(20)

plt.figure(figsize=(9, 7))
colors = ["#4C72B0" if f in SELECTED_FEATURES else "#B0B0B0" for f in top_features.index]
plt.barh(top_features.index[::-1], top_features.values[::-1], color=colors[::-1])
plt.xlabel("F-score (SelectKBest, f_regression)")
plt.title(f"Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ---------- VERSI A: SEMUA FITUR (baseline) ----------
scaler_X_all = MinMaxScaler()
scaler_y_all = MinMaxScaler()

X_train_all = scaler_X_all.fit_transform(train_X)
X_val_all   = scaler_X_all.transform(val_X)
X_test_all  = scaler_X_all.transform(test_X)
y_train_all = scaler_y_all.fit_transform(train_y.to_numpy().reshape(-1, 1)).ravel()
y_val_all   = scaler_y_all.transform(val_y.to_numpy().reshape(-1, 1)).ravel()
y_test_all  = scaler_y_all.transform(test_y.to_numpy().reshape(-1, 1)).ravel()

n_feat_all = X_train_all.shape[1]
X_train_all_3d = X_train_all.reshape(-1, 1, n_feat_all)
X_val_all_3d   = X_val_all.reshape(-1, 1, n_feat_all)
X_test_all_3d  = X_test_all.reshape(-1, 1, n_feat_all)

# ---------- VERSI B: FITUR TERPILIH (setelah SelectKBest) ----------
scaler_X_sel = MinMaxScaler()
scaler_y_sel = MinMaxScaler()

X_train_sel = scaler_X_sel.fit_transform(train_X_sel)
X_val_sel   = scaler_X_sel.transform(val_X_sel)
X_test_sel  = scaler_X_sel.transform(test_X_sel)
y_train_sel = scaler_y_sel.fit_transform(train_y.to_numpy().reshape(-1, 1)).ravel()
y_val_sel   = scaler_y_sel.transform(val_y.to_numpy().reshape(-1, 1)).ravel()
y_test_sel  = scaler_y_sel.transform(test_y.to_numpy().reshape(-1, 1)).ravel()

n_feat_sel = X_train_sel.shape[1]
X_train_sel_3d = X_train_sel.reshape(-1, 1, n_feat_sel)
X_val_sel_3d   = X_val_sel.reshape(-1, 1, n_feat_sel)
X_test_sel_3d  = X_test_sel.reshape(-1, 1, n_feat_sel)

print(f"Versi A (semua fitur)   : {n_feat_all} fitur")
print(f"Versi B (fitur terpilih): {n_feat_sel} fitur")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import gc

# Konfigurasi diambil langsung dari LSTM pertama (cell 15 & 16)
FIXED_PARAMS = {
    "units": 50,
    "dropout_rate": 0.2,
    "activation": "relu",
    "batch_size": 16,
}

def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)

def build_lstm_fixed(n_features):
    model = Sequential([
        Input(shape=(1, n_features)),
        LSTM(FIXED_PARAMS["units"], activation=FIXED_PARAMS["activation"]),
        Dropout(FIXED_PARAMS["dropout_rate"]),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")  # optimizer default, sama seperti LSTM pertama
    return model

def run_experiment(X_train_3d, y_train, X_val_3d, y_val, X_test_3d, y_test,
                    scaler_y, n_features, n_runs=5, label=""):
    results = []
    for run in range(n_runs):
        set_seed(run)
        tf.keras.backend.clear_session()
        gc.collect()

        model = build_lstm_fixed(n_features)
        early_stop = EarlyStopping(monitor="val_loss", patience=5,
                                    restore_best_weights=True, verbose=0)
        model.fit(X_train_3d, y_train, validation_data=(X_val_3d, y_val),
                  batch_size=FIXED_PARAMS["batch_size"], epochs=100,
                  callbacks=[early_stop], verbose=0, shuffle=False)

        y_pred_scaled = model.predict(X_test_3d, verbose=0).ravel()
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        results.append({"run": run, "rmse": rmse, "mae": mae})

        print(f"  [{label}] run {run+1}/{n_runs} -> RMSE: {rmse:.6f} | MAE: {mae:.6f}")

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    return pd.DataFrame(results)

In [ ]:
print("=== Menjalankan baseline (semua fitur) ===")
results_all = run_experiment(
    X_train_all_3d, y_train_all, X_val_all_3d, y_val_all,
    X_test_all_3d, y_test_all, scaler_y_all, n_feat_all,
    n_runs=5, label="baseline"
)

print("\n=== Menjalankan versi feature-selected ===")
results_sel = run_experiment(
    X_train_sel_3d, y_train_sel, X_val_sel_3d, y_val_sel,
    X_test_sel_3d, y_test_sel, scaler_y_sel, n_feat_sel,
    n_runs=5, label="selected"
)

print(f"\n=== Ringkasan ===")
print(f"Baseline (semua {n_feat_all} fitur)   -> RMSE: {results_all['rmse'].mean():.6f} ± {results_all['rmse'].std():.6f} | MAE: {results_all['mae'].mean():.6f} ± {results_all['mae'].std():.6f}")
print(f"Feature selected ({n_feat_sel} fitur)  -> RMSE: {results_sel['rmse'].mean():.6f} ± {results_sel['rmse'].std():.6f} | MAE: {results_sel['mae'].mean():.6f} ± {results_sel['mae'].std():.6f}")

In [ ]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(results_all["rmse"], results_sel["rmse"])
print(f"p-value (t-test, RMSE): {p_value:.4f}")

if p_value < 0.05:
    winner = "Feature Selection" if results_sel["rmse"].mean() < results_all["rmse"].mean() else "Semua Fitur"
    print(f"Perbedaan SIGNIFIKAN (p<0.05). Pemenang: {winner}")
else:
    print("Perbedaan TIDAK signifikan secara statistik — feature selection belum tentu terbukti meningkatkan performa dibanding baseline.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot([results_all["rmse"], results_sel["rmse"]],
                 labels=[f"Semua Fitur\n({n_feat_all})", f"Feature Selected\n({n_feat_sel})"])
axes[0].set_ylabel("RMSE (test set)")
axes[0].set_title(f"Distribusi RMSE — 5 run per versi\np-value = {p_value:.4f}")

means = [results_all["rmse"].mean(), results_sel["rmse"].mean()]
stds  = [results_all["rmse"].std(), results_sel["rmse"].std()]
bars = axes[1].bar(["Semua Fitur", "Feature Selected"], means, yerr=stds, capsize=6,
                    color=["#B0B0B0", "#4C72B0"])
axes[1].set_ylabel("RMSE rata-rata")
axes[1].set_title("Perbandingan Performa (RMSE)")
for bar, m in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, m, f"{m:.5f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

### hyperparam

In [ ]:
%pip install optuna

In [ ]:
import gc
import optuna
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

n_features = X_train_lstm.shape[1]


def build_lstm(units, dropout_rate, learning_rate, n_features):
  model = Sequential([
      Input(shape=(1, n_features)),
      # Menggunakan activation="tanh" agar kernel teroptimasi aktif
      LSTM(units, activation="tanh"),
      Dropout(dropout_rate),
      Dense(1),
  ])
  model.compile(
      optimizer=Adam(learning_rate=learning_rate), loss="mean_squared_error"
  )
  return model


def objective(trial):
  # Bersihkan memori graph dari trial sebelumnya
  tf.keras.backend.clear_session()
  gc.collect()

  units = trial.suggest_categorical("units", [32, 64, 100])
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.4)
  learning_rate = trial.suggest_float("learning_rate", 1e-3, 1e-2, log=True)
  # Gunakan batch_size yang lebih bersahabat untuk komputasi cepat
  batch_size = trial.suggest_categorical("batch_size", [64, 128])

  print(
      f"\n[Memulai Trial {trial.number}] units={units}, lr={learning_rate:.5f},"
      f" batch={batch_size}"
  )

  model = build_lstm(units, dropout_rate, learning_rate, n_features)

  early_stop = EarlyStopping(
      monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
  )

  history = model.fit(
      X_train_lstm_3d,
      y_train_lstm,
      validation_data=(X_val_lstm_3d, y_val_lstm),
      batch_size=batch_size,
      epochs=25,
      callbacks=[early_stop],
      verbose=1,  # Set 1 agar progres epoch per trial terlihat jelas
      shuffle=False,
  )

  best_val_loss = min(history.history["val_loss"])
  trial.set_user_attr("epochs_trained", len(history.history["val_loss"]))

  # Hapus instance model untuk membebaskan VRAM/RAM
  del model
  tf.keras.backend.clear_session()
  gc.collect()

  return best_val_loss


study = optuna.create_study(direction="minimize", study_name="lstm_tuning")

study.optimize(
    objective,
    n_trials=15,
    timeout=7200,
    show_progress_bar=True,
)

print("\n=== Hasil Tuning Optuna ===")
print(f"Jumlah trial selesai: {len(study.trials)}")
print(f"Best val_loss (MSE, scaled): {study.best_value:.8f}")
print(f"Best params: {study.best_params}")

In [ ]:
trials_df = study.trials_dataframe().sort_values("value").reset_index(drop=True)
print(trials_df[["number", "value", "params_units", "params_dropout_rate",
                  "params_learning_rate", "params_batch_size"]])

best_params = study.best_params

final_lstm = build_lstm(
    units=best_params["units"],
    dropout_rate=best_params["dropout_rate"],
    learning_rate=best_params["learning_rate"],
    n_features=n_features,
)

early_stop_final = EarlyStopping(
    monitor="val_loss",
    patience=5,              # boleh dikembalikan ke 5 di sini, karena ini training final (bukan tuning)
    restore_best_weights=True,
    verbose=1,
)

history_final = final_lstm.fit(
    X_train_lstm_3d, y_train_lstm,
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=best_params["batch_size"],
    epochs=100,
    callbacks=[early_stop_final],
    verbose=1,
    shuffle=False,
)

# Evaluasi di test set
y_pred_tuned_scaled = final_lstm.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_tuned_scaled = y_test_lstm.ravel()

y_pred_tuned = scaler_y_lstm.inverse_transform(y_pred_tuned_scaled.reshape(-1, 1)).ravel()
y_true_tuned = scaler_y_lstm.inverse_transform(y_true_tuned_scaled.reshape(-1, 1)).ravel()

mse_tuned  = mean_squared_error(y_true_tuned, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
mae_tuned  = mean_absolute_error(y_true_tuned, y_pred_tuned)

print(f"LSTM (tuned Optuna) -> MSE: {mse_tuned:.8f} | RMSE: {rmse_tuned:.8f} | MAE: {mae_tuned:.8f}")

In [ ]:
trials_df = study.trials_dataframe().sort_values("value").reset_index(drop=True)
print(trials_df[["number", "value", "params_units", "params_dropout_rate",
                  "params_learning_rate", "params_batch_size"]])

In [ ]:
best_params = study.best_params

final_lstm = build_lstm(
    units=best_params["units"],
    dropout_rate=best_params["dropout_rate"],
    learning_rate=best_params["learning_rate"],
    n_features=n_features,
)

early_stop_final = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history_final = final_lstm.fit(
    X_train_lstm_3d, y_train_lstm,
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=best_params["batch_size"],
    epochs=100,
    callbacks=[early_stop_final],
    verbose=1,
    shuffle=False,
)

# Evaluasi di test set
y_pred_tuned_scaled = final_lstm.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_tuned_scaled = y_test_lstm.ravel()

y_pred_tuned = scaler_y_lstm.inverse_transform(y_pred_tuned_scaled.reshape(-1, 1)).ravel()
y_true_tuned = scaler_y_lstm.inverse_transform(y_true_tuned_scaled.reshape(-1, 1)).ravel()

mse_tuned  = mean_squared_error(y_true_tuned, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
mae_tuned  = mean_absolute_error(y_true_tuned, y_pred_tuned)

print(f"LSTM (tuned Optuna) -> MSE: {mse_tuned:.8f} | RMSE: {rmse_tuned:.8f} | MAE: {mae_tuned:.8f}")

In [ ]:
import optuna.visualization.matplotlib as ov_mpl
import matplotlib.pyplot as plt

ov_mpl.plot_optimization_history(study)
plt.show()

ov_mpl.plot_param_importances(study)
plt.show()

# MODELLING SMUA JENIS

In [ ]:
# LSTM model
# Build LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

X_train_lstm_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_lstm_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_lstm_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

lstm_model2 = Sequential([
    Input(shape=(1, n_features)),
    LSTM(64, activation="relu", return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

lstm_model2.compile(optimizer="adam", loss="mean_squared_error")
lstm_model2.summary()

print(f"X_train_lstm: {X_train_lstm_3d.shape}")
print(f"X_val_lstm  : {X_val_lstm_3d.shape}")
print(f"X_test_lstm : {X_test_lstm_3d.shape}")

# training model
early_stop2 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training LSTM2 pada {len(X_train_lstm_3d):,} sample train "
      f"dan {len(X_val_lstm_3d):,} sample validation (maks 100 epoch) ...")

history2 = lstm_model2.fit(
    X_train_lstm_3d, y_train_lstm,
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop2],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_lstm2_scaled = lstm_model2.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_lstm2_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_lstm2 = scaler_y_lstm.inverse_transform(y_pred_lstm2_scaled.reshape(-1, 1)).ravel()
y_true_lstm2 = scaler_y_lstm.inverse_transform(y_true_lstm2_scaled.reshape(-1, 1)).ravel()

mse_lstm2  = mean_squared_error(y_true_lstm2, y_pred_lstm2)
rmse_lstm2 = np.sqrt(mse_lstm2)
mae_lstm2  = mean_absolute_error(y_true_lstm2, y_pred_lstm2)

print(f"LSTM2 -> MSE: {mse_lstm2:.8f} | RMSE: {rmse_lstm2:.8f} | MAE: {mae_lstm2:.8f}")

In [ ]:
#MLP
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

mlp_model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

mlp_model.compile(optimizer="adam", loss="mean_squared_error")
mlp_model.summary()

print(f"X_train_lstm: {X_train_lstm.shape}")
print(f"X_val_lstm  : {X_val_lstm.shape}")
print(f"X_test_lstm : {X_test_lstm.shape}")

# training model
early_stop_mlp = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training MLP pada {len(X_train_lstm):,} sample train "
      f"dan {len(X_val_lstm):,} sample validation (maks 100 epoch) ...")

history_mlp = mlp_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop_mlp],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_mlp_scaled = mlp_model.predict(X_test_lstm, verbose=0).ravel()
y_true_mlp_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_mlp = scaler_y_lstm.inverse_transform(y_pred_mlp_scaled.reshape(-1, 1)).ravel()
y_true_mlp = scaler_y_lstm.inverse_transform(y_true_mlp_scaled.reshape(-1, 1)).ravel()

mse_mlp  = mean_squared_error(y_true_mlp, y_pred_mlp)
rmse_mlp = np.sqrt(mse_mlp)
mae_mlp  = mean_absolute_error(y_true_mlp, y_pred_mlp)

print(f"MLP -> MSE: {mse_mlp:.8f} | RMSE: {rmse_mlp:.8f} | MAE: {mae_mlp:.8f}")

In [ ]:
#GRU
from tensorflow.keras.layers import GRU

n_features = X_train_lstm.shape[1]

X_train_gru_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_gru_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_gru_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

gru_model = Sequential([
    Input(shape=(1, n_features)),
    GRU(64, activation="relu", return_sequences=True),
    Dropout(0.2),
    GRU(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

gru_model.compile(optimizer="adam", loss="mean_squared_error")
gru_model.summary()

print(f"X_train_gru: {X_train_gru_3d.shape}")
print(f"X_val_gru  : {X_val_gru_3d.shape}")
print(f"X_test_gru : {X_test_gru_3d.shape}")

# training model
early_stop_gru = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training GRU pada {len(X_train_gru_3d):,} sample train "
      f"dan {len(X_val_gru_3d):,} sample validation (maks 100 epoch) ...")

history_gru = gru_model.fit(
    X_train_gru_3d, y_train_lstm,
    validation_data=(X_val_gru_3d, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop_gru],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_gru_scaled = gru_model.predict(X_test_gru_3d, verbose=0).ravel()
y_true_gru_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_gru = scaler_y_lstm.inverse_transform(y_pred_gru_scaled.reshape(-1, 1)).ravel()
y_true_gru = scaler_y_lstm.inverse_transform(y_true_gru_scaled.reshape(-1, 1)).ravel()

mse_gru  = mean_squared_error(y_true_gru, y_pred_gru)
rmse_gru = np.sqrt(mse_gru)
mae_gru  = mean_absolute_error(y_true_gru, y_pred_gru)

print(f"GRU -> MSE: {mse_gru:.8f} | RMSE: {rmse_gru:.8f} | MAE: {mae_gru:.8f}")

In [ ]:
# CNN 1D
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D

n_features = X_train_lstm.shape[1]

X_train_cnn_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_cnn_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_cnn_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

cnn_model = Sequential([
    Input(shape=(1, n_features)),
    Conv1D(64, kernel_size=1, activation="relu", padding="causal"),
    Conv1D(32, kernel_size=1, activation="relu", padding="causal"),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(1)
])

cnn_model.compile(optimizer="adam", loss="mean_squared_error")
cnn_model.summary()

print(f"X_train_cnn: {X_train_cnn_3d.shape}")
print(f"X_val_cnn  : {X_val_cnn_3d.shape}")
print(f"X_test_cnn : {X_test_cnn_3d.shape}")

# training model
early_stop_cnn = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training CNN1D pada {len(X_train_cnn_3d):,} sample train "
      f"dan {len(X_val_cnn_3d):,} sample validation (maks 100 epoch) ...")

history_cnn = cnn_model.fit(
    X_train_cnn_3d, y_train_lstm,
    validation_data=(X_val_cnn_3d, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop_cnn],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_cnn_scaled = cnn_model.predict(X_test_cnn_3d, verbose=0).ravel()
y_true_cnn_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_cnn = scaler_y_lstm.inverse_transform(y_pred_cnn_scaled.reshape(-1, 1)).ravel()
y_true_cnn = scaler_y_lstm.inverse_transform(y_true_cnn_scaled.reshape(-1, 1)).ravel()

mse_cnn  = mean_squared_error(y_true_cnn, y_pred_cnn)
rmse_cnn = np.sqrt(mse_cnn)
mae_cnn  = mean_absolute_error(y_true_cnn, y_pred_cnn)

print(f"CNN1D -> MSE: {mse_cnn:.8f} | RMSE: {rmse_cnn:.8f} | MAE: {mae_cnn:.8f}")

# SHAP

In [ ]:
%pip install shap

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

n_features = X_train_lstm_3d.shape[2]

# Fungsi wrapper: terima input 2D, reshape ke 3D untuk LSTM, kembalikan prediksi 1D
def model_predict(X_2d):
    X_3d = X_2d.reshape(-1, 1, n_features)
    return lstm_model.predict(X_3d, verbose=0).ravel()

# Reshape data ke 2D karena KernelExplainer butuh input tabular
X_train_2d = X_train_lstm_3d.reshape(X_train_lstm_3d.shape[0], n_features)
X_test_2d  = X_test_lstm_3d.reshape(X_test_lstm_3d.shape[0], n_features)

# Background sample - gunakan k-means summary agar lebih cepat (bukan raw sampling)
np.random.seed(42)
background_2d = shap.kmeans(X_train_2d, 30)  # ringkas jadi 30 "cluster center", jauh lebih cepat dari 100 raw sample

# Subset test set yang mau dijelaskan
n_explain = min(50, X_test_2d.shape[0])   # KernelExplainer lambat, mulai dari 50 dulu
X_explain_2d = X_test_2d[:n_explain]

explainer = shap.KernelExplainer(model_predict, background_2d)
shap_values_2d = explainer.shap_values(X_explain_2d, nsamples=100)

print(f"Shape SHAP values: {np.array(shap_values_2d).shape}")

In [ ]:
feature_names = FEATURE_COLS

plt.figure()
shap.summary_plot(shap_values_2d, X_explain_2d, feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot — LSTM")
plt.tight_layout()
plt.show()

plt.figure()
shap.summary_plot(shap_values_2d, X_explain_2d, feature_names=feature_names,
                   plot_type="bar", show=False)
plt.title("SHAP Feature Importance (mean |SHAP value|) — LSTM")
plt.tight_layout()
plt.show()

In [10]:
# Drop fitur temporal berdasarkan nama kolom
temporal_cols = ["year", "month", "day", "dayofweek", "quarter", "dayofyear", "weekofyear"]

assert len(FEATURE_COLS) == X_train_lstm.shape[1], "FEATURE_COLS tidak sinkron dengan X_train_lstm"

keep_idx = [i for i, c in enumerate(FEATURE_COLS) if c not in temporal_cols]
FEATURE_COLS_NT = [FEATURE_COLS[i] for i in keep_idx]

# Aman dilakukan setelah scaling: MinMaxScaler bekerja per kolom, jadi kolom lain tidak terpengaruh
X_train_nt = X_train_lstm[:, keep_idx]
X_val_nt   = X_val_lstm[:, keep_idx]
X_test_nt  = X_test_lstm[:, keep_idx]

n_features_nt = X_train_nt.shape[1]

X_train_nt_3d = X_train_nt.reshape(-1, 1, n_features_nt)
X_val_nt_3d   = X_val_nt.reshape(-1, 1, n_features_nt)
X_test_nt_3d  = X_test_nt.reshape(-1, 1, n_features_nt)

print(f"Fitur sebelum: {X_train_lstm.shape[1]} | sesudah drop temporal: {n_features_nt}")
print(f"Fitur tersisa: {FEATURE_COLS_NT}")

Fitur sebelum: 23 | sesudah drop temporal: 16
Fitur tersisa: ['ret_1d', 'ret_5d', 'ret_20d', 'ret_1d_lag1', 'ret_1d_lag2', 'ret_1d_lag3', 'vol_5d', 'vol_20d', 'volume_ratio', 'intraday_range', 'close_open', 'high_close', 'low_close', 'log_volume', 'ret_rank', 'volume_rank']


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
tf.random.set_seed(42)
tf.keras.backend.clear_session()

lstm_model_nt = Sequential([
    Input(shape=(1, n_features_nt)),
    LSTM(50, activation="relu"),
    Dropout(0.2),
    Dense(1)
])
lstm_model_nt.compile(optimizer="adam", loss="mean_squared_error")
lstm_model_nt.summary()

early_stop_nt = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history_nt = lstm_model_nt.fit(
    X_train_nt_3d, y_train_lstm,
    validation_data=(X_val_nt_3d, y_val_lstm),
    batch_size=16,
    epochs=100,
    callbacks=[early_stop_nt],
    verbose=1,
    shuffle=False,
)

# Evaluasi di test set (skala asli)
y_pred_nt_scaled = lstm_model_nt.predict(X_test_nt_3d, verbose=0).ravel()
y_pred_nt = scaler_y_lstm.inverse_transform(y_pred_nt_scaled.reshape(-1, 1)).ravel()
y_true_nt = scaler_y_lstm.inverse_transform(y_test_lstm.reshape(-1, 1)).ravel()

mse_nt  = mean_squared_error(y_true_nt, y_pred_nt)
rmse_nt = np.sqrt(mse_nt)
mae_nt  = mean_absolute_error(y_true_nt, y_pred_nt)

print(f"LSTM tanpa fitur temporal -> MSE: {mse_nt:.8f} | RMSE: {rmse_nt:.8f} | MAE: {mae_nt:.8f}")
print(f"LSTM awal (24 fitur)      -> MSE: {mse_lstm:.8f} | RMSE: {rmse_lstm:.8f} | MAE: {mae_lstm:.8f}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        13,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,451 (52.54 KB)

 Trainable params: 13,451 (52.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 159s 2ms/step - loss: 5.1595e-04 - val_loss: 4.9433e-04
Epoch 2/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 162s 2ms/step - loss: 4.0187e-04 - val_loss: 4.9624e-04
Epoch 3/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 409s 4ms/step - loss: 4.0173e-04 - val_loss: 4.9657e-04
Epoch 4/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 178s 2ms/step - loss: 4.0149e-04 - val_loss: 4.9511e-04
Epoch 5/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 256s 3ms/step - loss: 4.0129e-04 - val_loss: 4.9782e-04
Epoch 6/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 229s 2ms/step - loss: 4.0115e-04 - val_loss: 4.9823e-04
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.
LSTM tanpa fitur temporal -> MSE: 0.00048875 | RMSE: 0.02210772 | MAE: 0.01523360


NameError: name 'mse_lstm' is not defined